# 🎓 Antrenare Model Predicție Notă BAC

**Ce face:** Antrenează un model ML Ensemble care prezice nota la BAC pe baza exercițiilor rezolvate.

**Modele folosite:**
- Random Forest (200 arbori)
- Gradient Boosting (150 iterații)
- Neural Network (64→32→16 neuroni)
- **Ensemble Voting** = media celor 3

**Features:** 22 features extrase din exerciții (acuratețe, trend, consistență, etc.)

---
**Instrucțiuni:** Upload `synthetic_attempts.json` și `student_grades.json` în Kaggle, apoi **Run All**.

In [ ]:
# 1. Instalare dependințe
!pip install scikit-learn xgboost numpy -q

In [ ]:
# 2. Importuri
import json
import pickle
import numpy as np
from collections import defaultdict
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
)
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
    print('✅ XGBoost disponibil')
except:
    HAS_XGB = False
    print('⚠️ XGBoost indisponibil, mergem fără')

print('✅ Toate importurile OK')

In [ ]:
# 3. Încarcă datele (caută automat în toate folderele Kaggle)
import os

attempts_data = None
grades_data = None

def find_json(root, name):
    """Caută fișierul exact sau cu extensie greșită (ex: .jsonun)"""
    exact = os.path.join(root, name)
    if os.path.exists(exact):
        return exact
    # Caută variante cu extensie greșită
    base = name.replace('.json', '')
    for f in os.listdir(root):
        if f.startswith(base) and f.endswith(('.json', '.jsonun', '.jsonl')):
            return os.path.join(root, f)
    return None

# Caută în TOATE subfolderele din /kaggle/input/
for root, dirs, files in os.walk('/kaggle/input'):
    a_path = find_json(root, 'synthetic_attempts.json')
    g_path = find_json(root, 'student_grades.json')
    if a_path and g_path:
        with open(a_path) as f:
            attempts_data = json.load(f)
        with open(g_path) as f:
            grades_data = json.load(f)
        print(f'✅ Date găsite în: {root}')
        print(f'   attempts: {os.path.basename(a_path)}')
        print(f'   grades:   {os.path.basename(g_path)}')
        break

# Fallback local
if not attempts_data:
    for p in ['.', 'data', '/kaggle/working']:
        if not os.path.isdir(p):
            continue
        a_path = find_json(p, 'synthetic_attempts.json')
        g_path = find_json(p, 'student_grades.json')
        if a_path and g_path:
            with open(a_path) as f: attempts_data = json.load(f)
            with open(g_path) as f: grades_data = json.load(f)
            print(f'✅ Date găsite în: {p}')
            break

if not attempts_data:
    print('❌ NU GĂSESC DATELE!')
    print()
    print('📁 Fișiere în /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            print(f'   {os.path.join(root, f)}')
    raise FileNotFoundError('Adaugă dataset-ul și rulează din nou!')

print(f'📊 {len(attempts_data)} încercări, {len(grades_data)} studenți')

In [ ]:
# 4. Feature Engineering — extrage 22 de features din exerciții

def extract_features(attempts):
    """Extrage 22 features din lista de încercări ale unui student."""
    if not attempts:
        return np.zeros(22)
    
    total = len(attempts)
    correct = sum(1 for a in attempts if a['is_correct'])
    accuracy = correct / total
    
    # Acuratețe pe subiecte
    subjects = {1: [], 2: [], 3: []}
    for a in attempts:
        s = a.get('exercise_subject', 1)
        subjects[s].append(a['is_correct'])
    
    s1_acc = np.mean(subjects[1]) if subjects[1] else 0
    s2_acc = np.mean(subjects[2]) if subjects[2] else 0
    s3_acc = np.mean(subjects[3]) if subjects[3] else 0
    
    # Dificultate și timp
    diffs = [a.get('exercise_difficulty', 2) for a in attempts]
    times = [a.get('time_spent', 60) for a in attempts]
    avg_diff = np.mean(diffs)
    avg_time = np.mean(times)
    
    # Trend de învățare
    if total >= 10:
        first5 = np.mean([a['is_correct'] for a in attempts[:5]])
        last5 = np.mean([a['is_correct'] for a in attempts[-5:]])
        trend = last5 - first5
    else:
        trend = 0
    
    # Consistență
    if total >= 5:
        ws = min(5, total // 2)
        window_accs = [np.mean([a['is_correct'] for a in attempts[i:i+ws]]) 
                       for i in range(0, total - ws + 1, ws)]
        consistency = 1 - np.std(window_accs) if window_accs else 1
    else:
        consistency = 0.5
    
    # Performanță pe dificultate
    diff_perf = {1: [], 2: [], 3: [], 4: []}
    for a in attempts:
        diff_perf[a.get('exercise_difficulty', 2)].append(a['is_correct'])
    easy_acc = np.mean(diff_perf[1]) if diff_perf[1] else 0
    med_acc = np.mean(diff_perf[2]) if diff_perf[2] else 0
    hard_acc = np.mean(diff_perf[3]) if diff_perf[3] else 0
    expert_acc = np.mean(diff_perf[4]) if diff_perf[4] else 0
    
    # Time efficiency
    ct = [a.get('time_spent', 60) for a in attempts if a['is_correct']]
    wt = [a.get('time_spent', 60) for a in attempts if not a['is_correct']]
    avg_ct = np.mean(ct) if ct else 60
    avg_wt = np.mean(wt) if wt else 120
    time_eff = avg_ct / avg_wt if avg_wt > 0 else 1
    
    # Topic mastery
    topics = set(a.get('exercise_topic', 'x') for a in attempts)
    topic_div = len(topics) / 20
    topic_accs = defaultdict(list)
    for a in attempts:
        topic_accs[a.get('exercise_topic', 'x')].append(a['is_correct'])
    tm_scores = [np.mean(v) for v in topic_accs.values()]
    avg_tm = np.mean(tm_scores) if tm_scores else 0
    tm_var = np.std(tm_scores) if len(tm_scores) > 1 else 0
    
    # Streak
    cur_streak = max_streak = 0
    for a in attempts:
        if a['is_correct']:
            cur_streak += 1
            max_streak = max(max_streak, cur_streak)
        else:
            cur_streak = 0
    streak_ratio = max_streak / total
    
    # Recent performance
    recent = attempts[-10:] if total >= 10 else attempts
    recent_acc = np.mean([a['is_correct'] for a in recent])
    
    # Difficulty progression
    if total >= 10:
        first_diff = np.mean([a.get('exercise_difficulty', 2) for a in attempts[:total//2]])
        last_diff = np.mean([a.get('exercise_difficulty', 2) for a in attempts[total//2:]])
        diff_prog = last_diff - first_diff
    else:
        diff_prog = 0
    
    return np.array([
        total, accuracy, s1_acc, s2_acc, s3_acc, avg_diff, avg_time,
        trend, consistency, easy_acc, med_acc, hard_acc, expert_acc,
        time_eff, topic_div, avg_tm, tm_var, streak_ratio, max_streak,
        recent_acc, diff_prog, avg_ct
    ])

FEATURE_NAMES = [
    'total_attempts', 'overall_accuracy', 'subject1_acc', 'subject2_acc', 'subject3_acc',
    'avg_difficulty', 'avg_time', 'learning_trend', 'consistency',
    'easy_acc', 'medium_acc', 'hard_acc', 'expert_acc',
    'time_efficiency', 'topic_diversity', 'avg_topic_mastery', 'topic_mastery_variance',
    'streak_ratio', 'max_streak', 'recent_accuracy', 'difficulty_progression', 'avg_correct_time'
]

print(f'✅ {len(FEATURE_NAMES)} features definite')

In [ ]:
# 5. Pregătire dataset

# Grupează attempts pe student
students_attempts = defaultdict(list)
for a in attempts_data:
    students_attempts[a['user_id']].append(a)

X, y = [], []
for student in grades_data:
    uid = student['user_id']
    if uid in students_attempts:
        features = extract_features(students_attempts[uid])
        X.append(features)
        y.append(student['grade'])

X = np.array(X)
y = np.array(y)

print(f'✅ Dataset: {X.shape[0]} studenți × {X.shape[1]} features')
print(f'📊 Note: min={y.min():.1f}, max={y.max():.1f}, medie={y.mean():.1f}')

In [ ]:
# 6. Antrenare modele individuale + comparație

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print('=' * 60)

models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, max_depth=6, learning_rate=0.1, subsample=0.8, random_state=42),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(64, 32, 16), activation='relu', max_iter=500, random_state=42, early_stopping=True),
}

if HAS_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    cv_r2 = cross_val_score(model, X_scaled, y, cv=kfold, scoring='r2')
    cv_rmse = cross_val_score(model, X_scaled, y, cv=kfold, scoring='neg_root_mean_squared_error')
    results[name] = {'r2': cv_r2.mean(), 'rmse': -cv_rmse.mean()}
    print(f'🤖 {name:25s} | R²: {cv_r2.mean():.3f} ± {cv_r2.std():.3f} | RMSE: {-cv_rmse.mean():.3f}')

best = max(results.items(), key=lambda x: x[1]['r2'])
print(f'\n🏆 Cel mai bun: {best[0]} (R² = {best[1]["r2"]:.3f})')

In [ ]:
# 7. Antrenare ENSEMBLE final (media tuturor modelelor)

estimators = [
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(n_estimators=150, max_depth=6, learning_rate=0.1, subsample=0.8, random_state=42)),
    ('nn', MLPRegressor(hidden_layer_sizes=(64, 32, 16), activation='relu', max_iter=500, random_state=42, early_stopping=True)),
]
if HAS_XGB:
    estimators.append(('xgb', XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)))

ensemble = VotingRegressor(estimators=estimators)

print('🔄 Antrenare Ensemble...')
ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'\n✅ ENSEMBLE RESULTS:')
print(f'   RMSE: {rmse:.3f}')
print(f'   MAE:  {mae:.3f}')
print(f'   R²:   {r2:.3f}')

# Cross-validation pe ensemble
cv_r2 = cross_val_score(ensemble, X_scaled, y, cv=kfold, scoring='r2')
cv_rmse = cross_val_score(ensemble, X_scaled, y, cv=kfold, scoring='neg_root_mean_squared_error')
print(f'\n📊 Cross-Validation (5-fold):')
print(f'   R² Mean:   {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
print(f'   RMSE Mean: {-cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}')

In [ ]:
# 8. Feature Importance (de la Random Forest)

import matplotlib.pyplot as plt

# Antrenează RF separat pentru importanță
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
rf.fit(X_scaled, y)

importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(12, 6))
plt.bar(range(15), importances[sorted_idx], color='#58CC02')
plt.xticks(range(15), [FEATURE_NAMES[i] for i in sorted_idx], rotation=45, ha='right')
plt.title('Top 15 Feature Importance — Grade Predictor')
plt.ylabel('Importanță')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('\n🎯 Top 10 Features:')
for i in sorted_idx[:10]:
    print(f'   {FEATURE_NAMES[i]:30s} {importances[i]:.4f}')

In [ ]:
# 9. Predicții vs Valori Reale (grafic)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.6, c='#58CC02', edgecolors='#1a1a2e', s=60)
plt.plot([1, 10], [1, 10], 'r--', linewidth=2, label='Perfect')
plt.xlabel('Nota Reală', fontsize=14)
plt.ylabel('Nota Prezisă', fontsize=14)
plt.title(f'Predicție vs Realitate (R² = {r2:.3f})', fontsize=16)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('predictions_vs_real.png', dpi=150)
plt.show()

In [ ]:
# 10. Salvare model

save_data = {
    'model': ensemble,
    'scaler': scaler,
    'model_type': 'ensemble',
    'feature_names': FEATURE_NAMES,
    'training_metrics': {
        'rmse': rmse, 'mae': mae, 'r2': r2,
        'cv_r2_mean': cv_r2.mean(), 'cv_r2_std': cv_r2.std(),
        'cv_rmse_mean': -cv_rmse.mean(), 'cv_rmse_std': cv_rmse.std(),
    },
    'feature_importances': dict(zip(FEATURE_NAMES, importances)),
}

with open('grade_predictor_advanced.pkl', 'wb') as f:
    pickle.dump(save_data, f)

import os
size = os.path.getsize('grade_predictor_advanced.pkl') / 1024 / 1024
print(f'💾 Model salvat: grade_predictor_advanced.pkl ({size:.1f} MB)')
print(f'\n🎉 GATA! Descarcă grade_predictor_advanced.pkl și pune-l în backend/models/')

In [ ]:
# 11. Test rapid — simulare predicție

print('🧪 Test predicție pe studenți din test set:\n')
for i in range(min(5, len(X_test))):
    real = y_test[i]
    pred = y_pred[i]
    diff = abs(real - pred)
    emoji = '✅' if diff < 0.5 else '⚠️' if diff < 1.0 else '❌'
    print(f'{emoji} Student {i+1}: Nota reală={real:.1f}, Prezisă={pred:.1f} (diferență={diff:.2f})')